# **NOTEBOOK: Implementing SHAP into different blackbox models**

STEP 1: Find data and clean it

In [ ]:
import pandas as pd


from sklearn.model_selection import train_test_split

# Load the data
data = pd.read_csv('/Users/lauracohen/Desktop/Centrale 1A/Parcours recherche/mimic-iv-clinical-database-demo-2.2/hosp/admissions.csv')
data2 = pd.read_csv('/Users/lauracohen/Desktop/Centrale 1A/Parcours recherche/mimic-iv-clinical-database-demo-2.2/hosp/emar.csv')

data.head()

In [ ]:
#add binary death columns

data['death'] = data['deathtime'].notnull().astype(int)

data=data.drop(['discharge_location','edregtime', 'edouttime'], axis=1)


In [ ]:
data['admitted_hour'] = pd.to_datetime(data['admittime']).dt.hour

y= data.death
X=data.drop(["hospital_expire_flag","death"], axis=1)


In [ ]:
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)



for i in X_train_full.columns:
    if  X_train_full[i].dtype=="object": 
        print(i, X_train_full[i].nunique())

In [ ]:
import matplotlib.pyplot as plt
patient_counts = data.groupby("admit_provider_id")["subject_id"].nunique()
proportions = 100*patient_counts / patient_counts.sum()
proportions_sorted = proportions.sort_values(ascending=False)
patient_counts_sorted=patient_counts.sort_values(ascending=False)

patient_counts_sorted.head(20).plot(kind="bar")
plt.ylabel("number of Patients")
plt.show()

#we see that providers only tackle a few patients, the provider feature therefore must be ignored

In [ ]:
X_train_full=X_train_full.drop(["admit_provider_id"], axis=1)
X_train_full.head()

In [ ]:
features=["subject_id", "admission_type", "insurance", "language", "marital_status", "race", "admitted_hour"]
X_valid=X_valid_full[features].copy()
X_train=X_train_full[features].copy()

In [ ]:
X_train.loc[data["race"].str.contains("black", case=False, na=False), "race"] = "BLACK"
X_valid.loc[data["race"].str.contains("black", case=False, na=False), "race"] = "BLACK"
X_train.loc[data["race"].str.contains("hispanic", case=False, na=False), "race"] = "HISPANIC/LATINO"
X_valid.loc[data["race"].str.contains("hispanic", case=False, na=False), "race"] = "HISPANIC/LATINO"
X_train.loc[data["race"].str.contains("WHITE", case=False, na=False), "race"] = "WHITE"
X_valid.loc[data["race"].str.contains("WHITE", case=False, na=False), "race"] = "WHITE"
X_train.loc[data["race"].str.contains("UNABLE TO OBTAIN", case=False, na=False), "race"] = "UNKNOWN"
X_valid.loc[data["race"].str.contains("UNABLE TO OBTAIN", case=False, na=False), "race"] = "UNKNOWN"
X_train.loc[data["race"].str.contains("PORTUGUESE", case=False, na=False), "race"] = "WHITE"
X_valid.loc[data["race"].str.contains("PORTUGUESE", case=False, na=False), "race"] = "WHITE"
X_train.loc[data["race"].str.contains("PATIENT DECLINED TO ANSWER", case=False, na=False), "race"] = "UNKNOWN"
X_train.loc[data["race"].str.contains("OTHER", case=False, na=False), "race"] = "UNKNOWN"
X_train.loc[data["language"].str.contains(r"\?", case=False, na=False), "language"] = "UNKNOWN"
X_train.loc[data["language"].str.contains(r"\?", case=False, na=False), "language"] = "UNKNOWN"


display(X_valid.head())
display(X_train.head())




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score


categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[

        ('cat', categorical_transformer, [cname for  cname in X_train.columns if X_train[cname].dtype == "object"])
    ])
               

def test(n):
    model = RandomForestClassifier(n_estimators=n, random_state=44)

    my_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Fit
    my_pipeline.fit(X_train, y_train)

    # Predict probabilities for AUC
    preds = my_pipeline.predict_proba(X_valid)[:,1]

    # Evaluate
    return roc_auc_score(y_valid, preds)
    

In [ ]:
for i in range(50,600,50):
    print(i, test(i))

we pick 300



STEP 2: implementing SHAP:



In [ ]:
import eli5
from eli5.sklearn import PermutationImportance

my_pipeline.fit(X_train, y_train)

X_valid_transformed = my_pipeline.named_steps['preprocessor'].transform(X_valid)
if hasattr(X_valid_transformed, "toarray"):
    X_valid_transformed = X_valid_transformed.toarray()

trained_model = my_pipeline.named_steps['model']

perm = PermutationImportance(trained_model, random_state=1).fit(
    X_valid_transformed, y_valid)

eli5.show_weights(
    perm,
    feature_names=my_pipeline.named_steps['preprocessor'].get_feature_names_out()
)


here the permutation importance is mostly negative because all of the columns are categorical

In [ ]:
import shap
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def test(n, X_train, y_train, X_valid, y_valid, preprocessor):
    model = RandomForestClassifier(n_estimators=n, random_state=44)
    my_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    my_pipeline.fit(X_train, y_train)
    
    preds = my_pipeline.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, preds)
    
    trained_model = my_pipeline.named_steps['model']
    total_leaves = sum([tree.get_n_leaves() for tree in trained_model.estimators_])
    avg_leaves = total_leaves / n
    
    X_valid_transformed = my_pipeline.named_steps['preprocessor'].transform(X_valid)
    if hasattr(X_valid_transformed, "toarray"):
        X_valid_transformed = X_valid_transformed.toarray()
    
    explainer = shap.Explainer(trained_model, X_valid_transformed)
    shap_values = explainer(X_valid_transformed)
    
    feature_names = my_pipeline.named_steps['preprocessor'].get_feature_names_out()
    shap_importance = np.abs(shap_values.values[1]).mean(axis=0)
    
    importance_df = sorted(
        zip(feature_names, shap_importance),
        key=lambda x: x[1],
        reverse=True
    )

    shap.summary_plot(shap_values, features=X_valid_transformed, feature_names=feature_names, show=True)
    
    
    return {
        'auc': auc,
        'total_leaves': total_leaves,
        'avg_leaves': avg_leaves,
        'shap_importance': importance_df
    }


test(n=100, X_train=X_train, 
    y_train=y_train, 
    X_valid=X_valid, 
    y_valid=y_valid, 
    preprocessor=preprocessor)